# Train on a free GPU (Colab / Kaggle)

Why: training on the Mac (MPS) was slow and silently stalled for two epochs. On a T4 GPU this takes about 30-45 min per model.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.
Outputs (tables, figures, best model) are zipped at the end so you can download them and commit the small files to GitHub.

In [1]:
!git clone https://github.com/aadya197/Hate-Speech-Detection.git
%cd Hate-Speech-Detection
!pip install -q -r requirements.txt
!python -m src.data
!ls src

Cloning into 'Hate-Speech-Detection'...
remote: Enumerating objects: 331, done.
remote: Counting objects: 100% (331/331), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 331 (delta 96), reused 269 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (331/331), 4.18 MiB | 19.55 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/Hate-Speech-Detection

plain_text/hatexplain-train.parquet: downloading bytes:   0% 0.00/1.68M [00:00<?, ?B/s]
plain_text/hatexplain-train.parquet: downloading bytes: 100% 1.67M/1.67M [00:00<00:00, 1.86MB/s,  164kB/s  ]
plain_text/hatexplain-train.parquet: reconstructing file: 100% 1.68M/1.68M [00:00<00:00, 1.87MB/s,  165kB/s  ]
hatexplain-validation.parquet: 100% 212k/212k [00:00<00:00, 12.2MB/s]
hatexplain-test.parquet: 100% 212k/212k [00:00<00:00, 54.7MB/s]
Generating train split: 15383 examples [00:00, 251129.63 examples/s]
Generating validation split: 1922 examples [00:00, 234897.65 examples/s]
Generating test split: 192

In [ ]:
%cd /content/Hate-Speech-Detection
!git pull
!ls src

/content/Hate-Speech-Detection
Already up to date.
analysis.py	 data.py      __init__.py  run_ablation.py  xlmr.py
config.py	 evaluate.py  metrics.py   train_custom.py
custom_model.py  explain.py   __pycache__  train_xlmr.py


In [ ]:
!python -m src.run_ablation --stage design


>>> /usr/bin/python3 -m src.train_custom --tag hn_cls --fp16 --epochs 4 --head cls
Device: cuda | {'tag': 'hn_cls', 'encoder': 'bert-base-uncased', 'head': 'cls', 'rationale_lambda': 0.0, 'class_weights': False, 'freeze_layers': 0, 'max_len': 128, 'epochs': 4, 'lr': 2e-05, 'head_lr': 0.001, 'bs': 16, 'warmup_ratio': 0.1, 'seed': 42, 'fp16': True, 'smoke': False, 'save_model': False}
config.json: 100% 570/570 [00:00<00:00, 3.10MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 314kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.35MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.58MB/s]

model.safetensors: downloading bytes:  52% 229M/440M [00:01<00:00, 325MB/s, 17.7MB/s  ]
model.safetensors: reconstructing file:  15% 67.1M/440M [00:01<00:06, 54.5MB/s]
model.safetensors: downloading bytes:  72% 319M/440M [00:01<00:00, 254MB/s, 28.6MB/s  ]
model.safetensors: downloading bytes:  93% 410M/440M [00:01<00:00, 285MB/s, 33.8MB/s  ]
model.safetensors: reconstructing file:  76% 335M/44

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

Tesla T4


In [ ]:
# Execute this command to check the exact GPU hardware details and memory usage
!nvidia-smi

Tue Nov 10 11:59:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 1. Get the code (the repo must contain the latest src/, push from your Mac first)
!rm -rf Hate-Speech-Detection
!git clone https://github.com/aadya197/hate-speech-detection Hate-Speech-Detection
%cd Hate-Speech-Detection
!pip install -q "transformers>=4.45" datasets "accelerate>=1.0" scikit-learn seaborn matplotlib
# Safety check: Colab must have the newest script (it needs --fp16 and saves predictions)
!git log -1 --format="Latest commit: %h %s (%cd)"
!grep -q -- "--fp16" src/train_xlmr.py && grep -q save_predictions src/train_xlmr.py && echo "OK: scripts are up to date" || echo "STALE CLONE: push your latest src/ to GitHub first, then re-run this cell"

Cloning into 'Hate-Speech-Detection'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 170 (delta 53), reused 145 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (170/170), 1.31 MiB | 16.42 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/Hate-Speech-Detection
Latest commit: 2ef7986 Created using Colab (Tue Nov 10 17:29:40 2026 +0530)
OK: scripts are up to date


In [ ]:
# 2. Build the processed CSVs (downloads HateXplain, needs internet; about 1 minute)
!python -m src.data
!ls -la data/processed


plain_text/hatexplain-train.parquet: downloading bytes:  61% 1.03M/1.68M [00:00<00:00, 1.41MB/s]
plain_text/hatexplain-train.parquet: downloading bytes: 100% 1.67M/1.67M [00:00<00:00, 2.01MB/s,  163kB/s  ]
plain_text/hatexplain-train.parquet: reconstructing file: 100% 1.68M/1.68M [00:00<00:00, 2.02MB/s,  164kB/s  ]
hatexplain-validation.parquet: 100% 212k/212k [00:00<00:00, 8.02MB/s]
hatexplain-test.parquet: 100% 212k/212k [00:00<00:00, 9.85MB/s]
Generating train split: 15383 examples [00:00, 220012.13 examples/s]
Generating validation split: 1922 examples [00:00, 243283.81 examples/s]
Generating test split: 1924 examples [00:00, 253936.28 examples/s]
train: (15383, 6) -> /content/Hate-Speech-Detection/data/processed/hatexplain_train.csv
validation: (1922, 6) -> /content/Hate-Speech-Detection/data/processed/hatexplain_validation.csv
test: (1924, 6) -> /content/Hate-Speech-Detection/data/processed/hatexplain_test.csv
total 5192
drwxr-xr-x 2 root root    4096 Nov 10 12:00 .
drwxr-xr-x 3

In [ ]:
# 3. Quick sanity check: 1 tiny epoch, makes sure the whole pipeline works on this GPU
!python -m src.train_xlmr --model_name bert-base-uncased --tag bert --smoke --train_bs 16 --grad_accum 1 --fp16

## Model A: BERT (main model)
`bert-base-uncased` is the standard model for HateXplain and is English-only, which suits these posts.
Watch the log: `eval_macro_f1` should **change and grow** every epoch. If it prints the WARNING about identical metrics, stop.

In [ ]:
!python -m src.train_xlmr --model_name bert-base-uncased --tag bert --epochs 4 --train_bs 16 --grad_accum 1 --fp16

## Model B: XLM-RoBERTa again, now on GPU (for the comparison table)
Optional but recommended: the Mac run only really trained one epoch, so its 0.56 macro-F1 is not a fair number.
XLM-R is sensitive to the learning rate; 2e-5 with batch size 16 is a safe choice.

Uses a 10% learning-rate warmup (`--warmup_ratio 0.1`). The first epoch's `eval_macro_f1` should be well above 0.5; if it stays near 0.39, or the WARNING about identical metrics appears, stop the run.

In [ ]:
!python -m src.train_xlmr --model_name FacebookAI/xlm-roberta-base --tag xlmr_gpu --epochs 4 --train_bs 16 --grad_accum 1 --fp16 --warmup_ratio 0.1

Device: cuda
config.json: 100% 615/615 [00:00<00:00, 551kB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 87.2kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 64.1MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 85.3MB/s]

model.safetensors: downloading bytes:  10% 110M/1.12G [00:00<00:04, 205MB/s, 7.44MB/s  ] 
model.safetensors: downloading bytes:  67% 744M/1.12G [00:03<00:03, 120MB/s, 59.4MB/s  ]
model.safetensors: reconstructing file:  72% 804M/1.12G [00:06<00:02, 119MB/s, 6.54MB/s  ]
model.safetensors: downloading bytes: 100% 745M/745M [00:09<00:00, 77.9MB/s, 59.2MB/s  ]
model.safetensors: reconstructing file: 100% 1.12G/1.12G [00:09<00:00, 117MB/s, 78.9MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 12836.38it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.

In [ ]:
# 4. Show the results
import pandas as pd
for tag in ["bert", "xlmr_gpu"]:
    try:
        print(tag); display(pd.read_csv(f"results/tables/{tag}_metrics.csv")); display(pd.read_csv(f"results/tables/{tag}_per_class_test.csv"))
    except FileNotFoundError:
        print(tag, "not run yet")

bert
bert not run yet
xlmr_gpu


,split,accuracy,macro_precision,macro_recall,macro_f1
0,validation,0.688345,0.676344,0.681827,0.678209
1,test,0.693347,0.678382,0.684288,0.679837


,Class,Precision,Recall,F1-score,Support
0,Hate speech,0.728763,0.823232,0.773123,594
1,Normal,0.752632,0.731458,0.741894,782
2,Offensive,0.553753,0.498175,0.524496,548


In [ ]:
# 5. Zip ONLY this run's files (so unzipping on the Mac cannot overwrite your newer local results)
TAG = "xlmr_gpu"      # use "bert" if you re-run BERT
!zip -rq results_gpu.zip results/tables/{TAG}_* results/figures/{TAG}_* models/{TAG}_best
!ls -la results_gpu.zip
from google.colab import files   # on Kaggle: the zip appears under /kaggle/working instead
files.download("results_gpu.zip")

-rw-r--r-- 1 root root 842430040 Nov 10 12:13 results_gpu.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
for s in [1, 2]:
    !python -m src.train_custom --tag hn_cls_s{s} --fp16 --epochs 4 --head cls --seed {s}
    !python -m src.train_custom --tag hn_full_s{s} --fp16 --epochs 4 --head bilstm_attn --rationale_lambda 1.0 --seed {s}
    !python -m src.train_custom --tag hn_full_freeze6_s{s} --fp16 --epochs 4 --head bilstm_attn --rationale_lambda 1.0 --freeze_layers 6 --seed {s}

Device: cuda | {'tag': 'hn_cls_s1', 'encoder': 'bert-base-uncased', 'head': 'cls', 'rationale_lambda': 0.0, 'class_weights': False, 'freeze_layers': 0, 'max_len': 128, 'epochs': 4, 'lr': 2e-05, 'head_lr': 0.001, 'bs': 16, 'warmup_ratio': 0.1, 'seed': 1, 'fp16': True, 'smoke': False, 'save_model': False}
Loading weights: 100% 199/199 [00:00<00:00, 4832.48it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTE

In [ ]:
!zip -r hn_results.zip results/tables/hn_* results/tables/ablation.csv results/figures/hn_*
from google.colab import files
files.download("hn_results.zip")

  adding: results/tables/hn_attn_metrics.csv (deflated 35%)
  adding: results/tables/hn_attn_per_class_test.csv (deflated 30%)
  adding: results/tables/hn_attn_test_predictions.csv (deflated 56%)
  adding: results/tables/hn_attn_training_log.csv (deflated 45%)
  adding: results/tables/hn_attn_validation_predictions.csv (deflated 56%)
  adding: results/tables/hn_bilstm_metrics.csv (deflated 35%)
  adding: results/tables/hn_bilstm_per_class_test.csv (deflated 27%)
  adding: results/tables/hn_bilstm_test_predictions.csv (deflated 56%)
  adding: results/tables/hn_bilstm_training_log.csv (deflated 46%)
  adding: results/tables/hn_bilstm_validation_predictions.csv (deflated 56%)
  adding: results/tables/hn_cls_metrics.csv (deflated 35%)
  adding: results/tables/hn_cls_per_class_test.csv (deflated 30%)
  adding: results/tables/hn_cls_s1_metrics.csv (deflated 36%)
  adding: results/tables/hn_cls_s1_per_class_test.csv (deflated 32%)
  adding: results/tables/hn_cls_s1_test_predictions.csv (defla

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

After downloading, unzip into your project folder on the Mac. `models/` is git-ignored (large), but `results/tables` and `results/figures` should be committed.

In [ ]:
!python -m src.train_custom --tag hn_full --fp16 --epochs 4 --head bilstm_attn --rationale_lambda 1.0 --save_model

Device: cuda | {'tag': 'hn_full', 'encoder': 'bert-base-uncased', 'head': 'bilstm_attn', 'rationale_lambda': 1.0, 'class_weights': False, 'freeze_layers': 0, 'max_len': 128, 'epochs': 4, 'lr': 2e-05, 'head_lr': 0.001, 'bs': 16, 'warmup_ratio': 0.1, 'seed': 42, 'fp16': True, 'smoke': False, 'save_model': True}
config.json: 100% 570/570 [00:00<00:00, 2.13MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 203kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 18.5MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 33.8MB/s]

model.safetensors: downloading bytes:  92% 406M/440M [00:02<00:00, 330MB/s, 34.1MB/s  ]
model.safetensors: reconstructing file:  30% 134M/440M [00:02<00:04, 62.6MB/s]
model.safetensors: downloading bytes: 100% 415M/415M [00:04<00:00, 84.4MB/s, 37.8MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:04<00:00, 89.5MB/s, 37.2MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 18777.23it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key   

In [ ]:
!ls -lh /content/Hate-Speech-Detection/models

total 422M
-rw-r--r-- 1 root root 422M Nov 12 18:32 hn_full_best.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/Hate-Speech-Detection/models/hn_full_best.pt /content/drive/MyDrive/

Mounted at /content/drive


In [ ]:
import os
if not os.path.exists("/content/Hate-Speech-Detection"):
    !git clone https://github.com/aadya197/Hate-Speech-Detection.git /content/Hate-Speech-Detection
%cd /content/Hate-Speech-Detection
!git pull
!pip install -q -r requirements.txt
!python -m src.data
!ls src

Cloning into '/content/Hate-Speech-Detection'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (316/316), done.
remote: Compressing objects: 100% (274/274), done.
remote: Total 316 (delta 87), reused 261 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (316/316), 4.17 MiB | 8.22 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/Hate-Speech-Detection
Already up to date.

plain_text/hatexplain-train.parquet: downloading bytes:   0% 0.00/1.68M [00:00<?, ?B/s]
plain_text/hatexplain-train.parquet: downloading bytes: 100% 1.67M/1.67M [00:00<00:00, 2.37MB/s,  164kB/s  ]
plain_text/hatexplain-train.parquet: reconstructing file: 100% 1.68M/1.68M [00:00<00:00, 2.38MB/s,  165kB/s  ]
hatexplain-validation.parquet: 100% 212k/212k [00:00<00:00, 11.9MB/s]
hatexplain-test.parquet: 100% 212k/212k [00:00<00:00, 47.9MB/s]
Generating train split: 15383 examples [00:00, 252244.95 examples/s]
Generating validation split: 1922 examples [00:00, 228059.64 examples/s

In [ ]:
%cd /content/Hate-Speech-Detection
!python -m src.train_custom --tag hn_full --fp16 --epochs 4 --head bilstm_attn --rationale_lambda 1.0 --save_model
!python -m src.hn_faithfulness --tag hn_full --max_posts 300

/content/Hate-Speech-Detection
Device: cuda | {'tag': 'hn_full', 'encoder': 'bert-base-uncased', 'head': 'bilstm_attn', 'rationale_lambda': 1.0, 'class_weights': False, 'freeze_layers': 0, 'max_len': 128, 'epochs': 4, 'lr': 2e-05, 'head_lr': 0.001, 'bs': 16, 'warmup_ratio': 0.1, 'seed': 42, 'fp16': True, 'smoke': False, 'save_model': True}
config.json: 100% 570/570 [00:00<00:00, 2.91MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 302kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.64MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.72MB/s]

model.safetensors: downloading bytes:  88% 386M/440M [00:01<00:00, 284MB/s, 33.5MB/s  ]
model.safetensors: reconstructing file:  61% 268M/440M [00:01<00:01, 154MB/s]
model.safetensors: downloading bytes: 100% 415M/415M [00:04<00:00, 102MB/s, 37.8MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:04<00:00, 108MB/s, 36.2MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 14060.86it/s]
[transformers] BertModel LOAD REPORT fr

In [2]:
%cd /content/Hate-Speech-Detection
for s in [42, 1, 2]:
    !python -m src.train_custom --tag xlmr_cls_s{s} --encoder FacebookAI/xlm-roberta-base --head cls --fp16 --epochs 4 --lr 2e-5 --warmup_ratio 0.1 --seed {s}

/content/Hate-Speech-Detection
Device: cuda | {'tag': 'xlmr_cls_s42', 'encoder': 'FacebookAI/xlm-roberta-base', 'head': 'cls', 'rationale_lambda': 0.0, 'class_weights': False, 'freeze_layers': 0, 'max_len': 128, 'epochs': 4, 'lr': 2e-05, 'head_lr': 0.001, 'bs': 16, 'warmup_ratio': 0.1, 'seed': 42, 'fp16': True, 'smoke': False, 'save_model': False}
config.json: 100% 615/615 [00:00<00:00, 2.20MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 129kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 83.9MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 85.2MB/s]

model.safetensors: downloading bytes:  16% 177M/1.12G [00:01<00:05, 181MB/s, 14.6MB/s  ]
model.safetensors: downloading bytes:  20% 229M/1.12G [00:02<00:07, 124MB/s, 20.0MB/s  ]
model.safetensors: downloading bytes:  41% 462M/1.12G [00:03<00:02, 232MB/s, 38.4MB/s  ]
model.safetensors: downloading bytes:  44% 488M/1.12G [00:03<00:03, 188MB/s, 40.5MB/s  ]
model.safetensors: downloading bytes:  46% 518M/1.12G [00:0

In [3]:
!zip -r xlmr3.zip results/tables/xlmr_cls_* results/tables/ablation.csv
from google.colab import files
files.download("xlmr3.zip")

  adding: results/tables/xlmr_cls_s1_metrics.csv (deflated 35%)
  adding: results/tables/xlmr_cls_s1_per_class_test.csv (deflated 31%)
  adding: results/tables/xlmr_cls_s1_test_predictions.csv (deflated 56%)
  adding: results/tables/xlmr_cls_s1_training_log.csv (deflated 44%)
  adding: results/tables/xlmr_cls_s1_validation_predictions.csv (deflated 56%)
  adding: results/tables/xlmr_cls_s2_metrics.csv (deflated 36%)
  adding: results/tables/xlmr_cls_s2_per_class_test.csv (deflated 29%)
  adding: results/tables/xlmr_cls_s2_test_predictions.csv (deflated 56%)
  adding: results/tables/xlmr_cls_s2_training_log.csv (deflated 44%)
  adding: results/tables/xlmr_cls_s2_validation_predictions.csv (deflated 56%)
  adding: results/tables/xlmr_cls_s42_metrics.csv (deflated 35%)
  adding: results/tables/xlmr_cls_s42_per_class_test.csv (deflated 30%)
  adding: results/tables/xlmr_cls_s42_test_predictions.csv (deflated 56%)
  adding: results/tables/xlmr_cls_s42_training_log.csv (deflated 45%)
  addin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>